# Tutorial: Assessing Data Support in Predictive Optimization
## Final Google Colab companion notebook — annotated, section-by-section reproduction

**Purpose.** This self-contained notebook accompanies the IJAA tutorial manuscript. It recreates the worked regression/optimization examples, evaluates supporting evidence, reproduces the nine manuscript figures, and exports reviewable results. It explains **what each block does, why the calculation is needed, and how to interpret it**. All images are made from Python code; **no AI image-generation service is used**.

**Run in Google Colab:** upload **this single `.ipynb` file** to Colab, choose **Runtime → Run all**, and use the final download cell to obtain your figures and output files. There is **no need to upload a separate CSV or ZIP**: the original 150-row carpet benchmark, synthetic datasets, saved full-screen results, and underlying Python modules are embedded in a collapsed data cell. The notebook writes to `/content/IJAA_Tutorial_Colab` in Colab. In local Jupyter, it writes to `./IJAA_Tutorial_Colab`.

> **Scientific scope:** The carpet benchmark consists of **generated observations** associated with a proprietary manufacturing process; it is **not a record of validated production runs**. Synthetic response laws are stipulated and used only to generate data and evaluate decisions. The true response is **never given to the predictor or optimizer**. Every modeled cost, risk, and feasibility finding must be interpreted in the context stated in the paper.

### Correspondence to the manuscript

| Paper location | Main task in this notebook | Exhibits |
|:--|:--|:--|
| §2, Equations (1)–(4) | Fitted response, decision model, box/hull support | **Figure 1** |
| §§3.1–3.2, Appendix A1 | Original 16-setting interior-gap example | **Figure 2, Table 1** |
| §3.3, Appendix A1, §7.1 | Five targeted samples and one linear refit | **Figure 8** |
| §4, Equation (5) and (6) | Three-input carpet regression, costs, joint support and proximity | **Figures 3–4, Table 2** |
| §5.1, Appendix A2 | Constant margin / prediction-interval sensitivity | **Figure 5** |
| §5.2, Appendix A3 | Matched synthetic response laws and 100 noisy repetitions | **Figure 6, Table 3** |
| §6, Appendix A4 | 49 setting-omission sensitivity checks, fixed predictor | **Figure 7** |
| §7.2 | Carpet candidate decision record and validation plan | **Figure 9** |

**Reproduction modes.** The default path **recomputes the central small examples** and reads the archived 816-run simulation results for the larger Monte Carlo figures/table, checking their declared counts. An **optional full-screen cell** at the end reruns all 16,320 scenario optimization comparisons from seeds 100–199. The full-screen run is *off by default* to make ordinary Colab use practical. These modes are distinguished explicitly so precomputed data are not represented as freshly regenerated simulations.

## 0. Environment and package setup

**Sections supported:** all sections and Appendices A1–A4.

This cell checks for the required libraries and installs only packages missing from the Python environment. NumPy/SciPy handle regression and mathematical programming, pandas manages the results, Matplotlib generates Figures 1–7, and `python-pptx` constructs **editable PowerPoint shapes** for Figures 8–9. It prints library versions to aid reproducibility; your installed versions may differ from the original analysis environment.

In [ ]:
# Step 0.1 — Import standard Python tools before touching any analysis data.
import importlib.util
import os
import platform
import subprocess
import sys
from pathlib import Path

# Step 0.2 — Add only missing dependencies; do not force upgrades of Colab's stock stack.
required_modules = {
    'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
    'matplotlib': 'matplotlib', 'PIL': 'Pillow', 'pptx': 'python-pptx'
}
missing = [distribution for module, distribution in required_modules.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing Python packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

# Step 0.3 — Import analytical libraries only after dependency resolution.
import base64
import hashlib
import io
import json
import runpy
import shutil
import zipfile
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import linprog
from scipy.spatial import ConvexHull
from scipy.stats import t
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Image

IN_COLAB = bool(os.environ.get('COLAB_RELEASE_TAG')) or ('google.colab' in sys.modules) or (importlib.util.find_spec('google.colab') is not None)
WORK = Path('/content/IJAA_Tutorial_Colab' if IN_COLAB
            else Path.cwd() / 'IJAA_Tutorial_Colab')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)  # Keep all relative paths stable for every following code cell.
print('Python:', platform.python_version(), '| Colab:', IN_COLAB)
print('numpy:', np.__version__, '| pandas:', pd.__version__, '| scipy:', scipy.__version__)
print('Working directory:', WORK)

## 0.1 Embedded study files — no manual upload required

The next **collapsed, machine-generated** cell contains a ZIP archive encoded as text. It includes source code and the exact inputs/results used in the paper. The purpose of this unusual format is to let you download and run **one `.ipynb` file** in Google Colab without uploading any additional data files or fetching files from a website.

- The 150-row carpet CSV is stored unchanged in `original_baseline/data/`.
- The eight synthetic configurations, original generated datasets, and 100-noise-repetition summaries reside in `synthetic_study/`.
- The analytical source modules (including the full prespecified scenario engine) are in `code/`.
- High-resolution previews of Figures 8 and 9 are included; the next sections also regenerate **editable PowerPoint versions** in Python.

**Do not edit the encoded payload.** The code checks its SHA-256 hash and refuses to continue if it was corrupted. This is a data transport cell, not an analysis method.

In [ ]:
# EMBEDDED STUDY ASSETS — generated from the validated electronic supplement.
# This large text block is the ZIP file's Base64 representation, not executable analysis.
# Please keep it collapsed while reading the notebook.
_EMBEDDED_ZIP_BASE64 = """
...
"""
EXPECTED_ZIP_SHA256 = '62dab38ab76e282ad0e01db3ea77b7d98eab1e449d534f6c52299cab87dae7b2'

# Decode and check the exact bytes before extracting anything.
raw_zip = base64.b64decode(''.join(_EMBEDDED_ZIP_BASE64.split()))
actual_digest = hashlib.sha256(raw_zip).hexdigest()
assert actual_digest == EXPECTED_ZIP_SHA256, 'Embedded study assets failed SHA-256 verification.'

# Extract safely: reject any absolute path or '../' traversal in the archive.
with zipfile.ZipFile(io.BytesIO(raw_zip)) as archive:
    for member in archive.infolist():
        destination = (WORK / member.filename).resolve()
        if destination != WORK and WORK not in destination.parents:
            raise ValueError('Unsafe archive path: ' + member.filename)
    archive.extractall(WORK)
    print('Verified and extracted', len(archive.namelist()), 'study files.')

# The core engine and PPTX-building scripts are now importable from code/.
source_dir = str(WORK / 'code')
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
FIGURES = WORK / 'figures_v2'
FIGURES.mkdir(exist_ok=True)
RESULTS = WORK / 'recomputed_results'
RESULTS.mkdir(exist_ok=True)
print('Inputs:', WORK / 'original_baseline/data/carpet_simulated_data.csv')
print('Exports:', FIGURES)

## 1. Decision framework and historical-domain checks
**Manuscript:** §2, Equations (1)–(4), Figure 1.

The first step is to distinguish the **fitted response** from the **optimization problem**. The paper minimizes modeled ingredient cost subject to a required prediction and other business constraints. A historical **box** enforces separate observed minima/maxima; a **convex hull** enforces shared weights on *complete observed input combinations*. That difference is central to the tutorial and does **not** imply that hull membership guarantees true performance.

Below, the original verified Python LP routine is placed directly in the notebook so readers can audit the constraints. For the three-input carpet benchmark, this routine uses SciPy's HiGHS linear-programming solver and applies the same numerical checks to each candidate.

In [ ]:
# Step 1.1 — Exact linear-regression/optimization functions used by the baseline notebook.
# `fit_ols`: estimates an intercept and numerical-input slopes from observed X and y.
# `solve_lp`: uses the same fitted coefficients across comparisons; only the domain changes.
# `full_hull_membership`: checks whether one *common* nonnegative weight vector can
# reconstruct all coordinates of a proposed setting (a joint, not marginal, check).
# `solution_row`: formats the returned solution without changing numerical precision.
#
# NOTE: A solution that satisfies the mathematical constraints is only a *modeled*
# candidate. It is not proof that a chemical recipe is physically admissible.
CHECK_ATOL = 1e-7
SOLVER_OPTIONS = {'primal_feasibility_tolerance': 1e-9,
                  'dual_feasibility_tolerance': 1e-9}
COST_CHANGE_TOL = 1e-7

def fit_ols(inputs, response):
    Z = np.column_stack([np.ones(len(inputs)), inputs])
    beta, _, rank, _ = np.linalg.lstsq(Z, response, rcond=None)
    if rank != Z.shape[1]:
        raise ValueError('The regression design is not full rank.')
    return beta, Z

def solve_lp(beta, costs, observed, goal, domain, target, budget,
             domain_points=None, fixed_box=None):
    beta = np.asarray(beta, dtype=float)
    costs = np.asarray(costs, dtype=float)
    observed = np.asarray(observed, dtype=float)
    if goal not in ('max_response', 'min_cost'):
        raise ValueError('Unknown objective.')
    objective = -beta[1:] if goal == 'max_response' else costs
    A = np.vstack([-beta[1:], costs])
    b = np.array([beta[0] - target, budget], dtype=float)
    weights, points = None, None
    if domain == 'hull':
        points = np.unique(observed if domain_points is None else domain_points, axis=0)
        A_hull = np.vstack([A @ points.T, -points.T])
        b_hull = np.r_[b, np.zeros(observed.shape[1])]
        if fixed_box is not None:
            original_lower, original_upper = fixed_box
            A_hull = np.vstack([A_hull, points.T, -points.T])
            b_hull = np.r_[b_hull, original_upper, -original_lower]
        result = linprog(points @ objective, A_ub=A_hull, b_ub=b_hull,
                         A_eq=np.ones((1, len(points))), b_eq=[1.0],
                         bounds=(0, None), method='highs', options=SOLVER_OPTIONS)
    else:
        if domain == 'box':
            bounds = list(zip(np.maximum(0, observed.min(axis=0)), observed.max(axis=0)))
        elif domain == 'no_history':
            bounds = [(0, None)] * observed.shape[1]
        else:
            raise ValueError('Unknown domain.')
        result = linprog(objective, A_ub=A, b_ub=b, bounds=bounds,
                         method='highs', options=SOLVER_OPTIONS)
    if not result.success:
        if result.status == 2:
            return {'feasible': False, 'status': int(result.status), 'settings': None,
                    'prediction': np.nan, 'cost': np.nan, 'weights': None, 'points': points}
        raise RuntimeError('Unresolved optimization: ' + result.message)
    if domain == 'hull':
        weights = result.x
        settings = weights @ points
        assert abs(weights.sum() - 1) <= CHECK_ATOL and weights.min() >= -CHECK_ATOL
    else:
        settings = result.x
    prediction = float(beta[0] + beta[1:] @ settings)
    cost = float(costs @ settings)
    assert settings.min() >= -CHECK_ATOL
    assert prediction >= target - CHECK_ATOL and cost <= budget + CHECK_ATOL
    if fixed_box is not None:
        assert np.all(settings >= fixed_box[0] - CHECK_ATOL)
        assert np.all(settings <= fixed_box[1] + CHECK_ATOL)
    return {'feasible': True, 'status': int(result.status), 'settings': settings,
            'prediction': prediction, 'cost': cost, 'weights': weights, 'points': points}

def solution_row(goal, domain, solution):
    return {'goal': goal, 'domain': domain, 'feasible': solution['feasible'],
            **{f'x{k+1}': value for k, value in enumerate(solution['settings'])},
            'prediction': solution['prediction'], 'modeled_cost': solution['cost']}

def full_hull_membership(points, candidate):
    points = np.unique(np.asarray(points), axis=0)
    lower = points.min(axis=0)
    scale = np.ptp(points, axis=0)
    scale = np.where(scale > 0, scale, 1.0)
    scaled_points = (points - lower) / scale
    scaled_candidate = (np.asarray(candidate) - lower) / scale
    result = linprog(np.zeros(len(points)),
                     A_eq=np.vstack([np.ones(len(points)), scaled_points.T]),
                     b_eq=np.r_[1.0, scaled_candidate], bounds=(0, None),
                     method='highs', options=SOLVER_OPTIONS)
    if result.status not in (0, 2):
        raise RuntimeError('Unresolved membership check: ' + result.message)
    if result.success:
        assert np.allclose(result.x @ points, candidate, atol=CHECK_ATOL, rtol=0)
    return bool(result.success)

### 1.1 Load and inspect the actual benchmark provided with the paper
**Manuscript:** §4, Appendix A4.

The carpet data have **150 generated response records at 49 distinct three-chemical settings**. We preserve repeated settings for regression fitting but deduplicate the complete input vectors when building the hull. Chemical units and operating limits are proprietary/unknown; observed ranges must not be presented as established engineering limits.

In [ ]:
# Step 1.2 — Load the untouched carpet benchmark from the embedded archive.
CARPET_FILE = WORK / 'original_baseline/data/carpet_simulated_data.csv'
carpet = pd.read_csv(CARPET_FILE)
FEATURES = ['x1', 'x2', 'x3']
X = carpet[FEATURES].to_numpy(dtype=float)
y = carpet['y'].to_numpy(dtype=float)
unique_X = np.unique(X, axis=0)  # Different from keeping all 150 response records!

# A row-level integrity check matters here: losing repeats changes the regression.
assert len(carpet) == 150
assert len(unique_X) == 49
print(f'Carpet response records: {len(carpet)}')
print(f'Distinct full ingredient settings: {len(unique_X)}')
print('Observed individual input ranges (not physical limits):')
display(carpet[FEATURES].agg(['min', 'max']).T)

### 1.2 Fitted regression and least-cost candidate models
**Manuscript:** §4, Equation (5), Table 2.

We fit the three-input response model **once** to all 150 generated observations, then hold its coefficients constant while solving three optimization formulations. The *no-history* optimum is diagnostic only because it chooses zero doses for two chemicals even though the process requires all three. The *box* and *hull* optima are still unvalidated formulations, not tested production recipes.

In [ ]:
# Step 1.3 — Estimate the main-effects model on *every* response record.
carpet_beta, design_matrix = fit_ols(X, y)
residuals = y - design_matrix @ carpet_beta
n, p = design_matrix.shape
carpet_r2 = 1 - (residuals @ residuals) / np.sum((y - y.mean())**2)
carpet_r2_adjusted = 1 - (1-carpet_r2) * (n-1)/(n-p)

# The business model uses recorded-unit costs and the stated quality/budget limits.
carpet_costs = np.array([2.56, 2.50, 21.52])
quality_target = 80.0
budget = 1000.0
original_lower, original_upper = X.min(axis=0), X.max(axis=0)
original_ranges = original_upper - original_lower

carpet_solutions = {}
for domain in ['no_history', 'box', 'hull']:
    # Under the common fitted model, changes in optimum arise from domain changes.
    sol = solve_lp(carpet_beta, carpet_costs, X, 'min_cost', domain,
                   target=quality_target, budget=budget)
    assert sol['feasible'], f'{domain} baseline unexpectedly infeasible'
    carpet_solutions[domain] = sol

# For display we round only here, never in calculations or constraint checks.
carpet_rows = [
    {'formulation': domain, **{f'x{i+1}': float(v) for i,v in enumerate(sol['settings'])},
     'predicted_stability': sol['prediction'], 'modeled_cost': sol['cost']}
    for domain,sol in carpet_solutions.items()
]
carpet_baseline_table = pd.DataFrame(carpet_rows)
display(carpet_baseline_table.round(5))
print('OLS coefficients:', carpet_beta)
print('R^2:', round(carpet_r2, 6), '| adjusted R^2:', round(carpet_r2_adjusted, 6))

# Reproduce the manuscript's benchmark before moving to additional checks.
assert np.isclose(carpet_r2, 0.988708, atol=5e-6)
assert np.isclose(carpet_solutions['box']['cost'], 32.17, atol=.01)
assert np.isclose(carpet_solutions['hull']['cost'], 33.27, atol=.01)

### 1.3 Verify the full three-dimensional hull rather than only pairwise projections
**Manuscript:** §2, Equation (4); §4, Equation (6), Figure 3.

The box recommendation satisfies every individual observed range but lies outside the full hull. The hull recommendation has nonnegative weights on observed *complete* recipes, and the same weight applies to every input coordinate. Such a certificate is geometric; it is not a chemical-mixing or response-validity certificate.

In [ ]:
# Step 1.4 — Check the actual recommended input vectors against joint support.
box_x = carpet_solutions['box']['settings']
hull_x = carpet_solutions['hull']['settings']
box_in_hull = full_hull_membership(unique_X, box_x)
hull_in_hull = full_hull_membership(unique_X, hull_x)
assert not box_in_hull and hull_in_hull

# Show the few historical settings needed to reconstruct the hull optimum.
hull_solution = carpet_solutions['hull']
witness = pd.DataFrame(hull_solution['points'], columns=FEATURES)
witness['shared_weight'] = hull_solution['weights']
witness = witness[witness.shared_weight > 1e-7].copy()
assert np.allclose(witness[FEATURES].to_numpy().T @ witness.shared_weight.to_numpy(),
                   hull_x, atol=1e-7)
print('Box inside full hull:', box_in_hull)
print('Hull optimum inside full hull:', hull_in_hull)
print('Shared-weight reconstruction (Equation 6):')
display(witness.round(8))

### Figure 1 — The complete decision-evidence workflow
**Manuscript:** §2, Figure 1.

The workflow starts with observed data, moves through a fitted model to an optimized *candidate*, and then forces three separate checks: (1) complete input support, (2) response uncertainty, and (3) sensitivity to the historical support boundary. Its endpoint is a **conditional action and specific next evidence**, not automatic implementation.

The next code cell defines a consistent, high-contrast Matplotlib style for Figures 1–7. Later plotting cells reuse it, so all exported titles, axis labels, and tick labels are black and the figures remain readable at journal scale.

In [ ]:
# Step 1.5 — Shared color palette, font settings and PNG/SVG exporter.
# Source: reproducible IJAA visual notebook; generated figures use no AI images.
# EXPORT RULE: save one high-resolution PNG for Word and one vector SVG for editing.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Polygon
from matplotlib.lines import Line2D
from scipy.spatial import ConvexHull

ROOT = Path.cwd()
OUT = ROOT / 'figures_v2'
OUT.mkdir(exist_ok=True)
T = ROOT / 'manuscript_exhibits' / 'tables'
O = ROOT / 'original_baseline' / 'results'
P = ROOT / 'original_baseline' / 'data'
S = ROOT / 'selected_datasets'
INK='#000000'; TEAL='#0E6870'; BLUE='#265D89'; RUST='#9E472B'; GOLD='#8C6328'
MUTED=INK; LIGHT='#EEF3F5'; GRID='#DDE3E6'; WHITE='#FFFFFF'
plt.rcParams.update({
    'font.family':'DejaVu Sans', 'font.size':11.0, 'axes.labelsize':11.0,
    'axes.titlesize':11.5, 'axes.titleweight':'semibold', 'axes.titlecolor':INK,
    'xtick.labelsize':10.0, 'ytick.labelsize':10.0, 'legend.fontsize':10.0,
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.edgecolor':INK, 'axes.labelcolor':INK, 'text.color':INK,
    'xtick.color':INK, 'ytick.color':INK,
    'figure.facecolor':'white', 'axes.facecolor':'white', 'savefig.facecolor':'white',
    'pdf.fonttype':42, 'svg.fonttype':'none',
})
def export(fig, name):
    fig.savefig(OUT / (name+'.png'), dpi=300, bbox_inches='tight', pad_inches=0.15)
    fig.savefig(OUT / (name+'.svg'), bbox_inches='tight', pad_inches=0.15)
    plt.close(fig)
    print(name, (OUT / (name+'.png')).stat().st_size, 'bytes')

In [ ]:
# Step 1.6 — Draw and export Figure 1 using native Matplotlib rectangles and arrows.
# Scientific meaning: optimization yields a decision candidate, not a validated action.
# All boxes are dimensioned so their text remains inside the borders.

# Figure 1: all text stays inside its box; black text on white or very pale fills.
fig, ax = plt.subplots(figsize=(7.7,4.45))
ax.set_xlim(0,100); ax.set_ylim(0,100); ax.axis('off')
def block(x,y,w,h,title,sub='',edge='#8197A3',fill=WHITE,fw=10.1,sub_fs=9.5):
    patch=FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.25,rounding_size=2',
                         linewidth=1.55,edgecolor=edge,facecolor=fill)
    ax.add_patch(patch)
    ax.text(x+w/2,y+h*.67,title,ha='center',va='center',fontsize=fw,
            fontweight='bold',color=INK,linespacing=1.07,clip_on=True)
    if sub:
        ax.text(x+w/2,y+h*.24,sub,ha='center',va='center',fontsize=sub_fs,color=INK,
                linespacing=1.04,clip_on=True)
    return patch

def arrow(a,b,color='#8197A3',lw=1.55):
    ax.add_patch(FancyArrowPatch(a,b,arrowstyle='-|>',mutation_scale=12,
                                 linewidth=lw,color=color,shrinkA=0,shrinkB=0))

block(1,75,30,18,'Observed inputs\n+ outcomes','Historical evidence',fw=9.9,sub_fs=9.0)
block(35,75,30,18,'Fitted response\nmodel','Estimate outcome at x',fw=10.0,sub_fs=9.0)
block(69,75,30,18,'Optimize the\nbusiness goal','Produce candidate x*',fw=10.0,sub_fs=9.0)
arrow((31.3,84),(34.6,84)); arrow((65.3,84),(68.6,84))
arrow((84,74.7),(84,67.0))
ax.text(50,62,'Check the candidate, not only the fitted model',ha='center',va='center',
        fontsize=11.2,fontweight='bold',color=INK)
for x,header,query in [
    (1,'JOINT INPUT\nSUPPORT','Within individual ranges?\nNear complete records?'),
    (35,'RESPONSE\nUNCERTAINTY','What response matters?\nCost of the allowance?'),
    (69,'BOUNDARY\nSENSITIVITY','Which records define it?\nDoes the answer change?')]:
    patch=FancyBboxPatch((x,29),30,25,boxstyle='round,pad=0.28,rounding_size=2',
                        linewidth=1.6,edgecolor=TEAL,facecolor='#F5F9F9')
    ax.add_patch(patch)
    ax.text(x+15,47.0,header,ha='center',va='center',fontsize=9.3,
            fontweight='bold',color=INK)
    ax.text(x+15,36.6,query,ha='center',va='center',fontsize=9.5,
            linespacing=1.42,color=INK)
    arrow((x+15,28.7),(x+15,22.4))
block(10,4,80,16,'Conditional decision + next evidence',
      'Document modeled cost, unresolved concern, and justified follow-up',
      edge=BLUE,fill='#F5F8FB',fw=10.4,sub_fs=9.2)
# Connections terminate on the top edge of the outcome box, without a crossing line.
export(fig,'Figure_01_Evidence_Workflow')

## 2. Unsampled interior counterexample
**Manuscript:** §§3.1–3.2; Appendix A1; Figure 2 and Table 1.

Here 16 unique input settings lie on a circle around (0.5, 0.5), with three identical noiseless responses per location (48 response records). The known mean response has a localized interior dip that does not appear at any training point. Because the fit sees only responses on the circle, ordinary least squares recovers the linear trend perfectly. The solver uses the fitted trend and thus recommends the **unsampled center** even though the stipulated true response there is too low.

We are allowed to examine the known generating function *after optimization* because this is a deliberately constructed synthetic example. That oracle distinction is essential to the tutorial's logic.

In [ ]:
# Step 2.1 — Import the prespecified data/response/optimizer engine.
# This source resides inside this notebook's verified embedded ZIP.
# `true_mean` is only called to EVALUATE model-selected decisions, never to fit them.
import synthetic_engine as engine

ring = engine.generate_dataset('interior_d12', noise_sd=0.0, seed=engine.REPRESENTATIVE_SEED)
ring_fit = engine.fit_ols(ring['X'], ring['y'])
ring_solutions = {
    name: engine.solve_model(ring_fit, ring['X'], 'interior_d12', domain=name, allowance=0.0)
    for name in ('no_history','box','hull','observed')
}
ring_rows = []
for name, sol in ring_solutions.items():
    if not sol['feasible']:
        continue
    # The truth is deliberately evaluated ONLY AFTER the choice has been optimized.
    known_mean = float(engine.true_mean('interior_d12', sol['x']))
    ring_rows.append({'historical_domain': name, 'x1': sol['x'][0], 'x2': sol['x'][1],
                      'modeled_cost': sol['cost'], 'fitted_quality': sol['prediction'],
                      'known_synthetic_quality': known_mean})
ring_comparison = pd.DataFrame(ring_rows)
display(ring_comparison.round(4))
print('Training rows:', len(ring['X']), 'distinct settings:', len(np.unique(ring['X'],axis=0)))
print('Training R^2:', ring_fit['r_squared'])

center = ring_solutions['hull']['x']
assert np.allclose(center, [0.5, 0.5], atol=1e-7)
assert np.isclose(engine.true_mean('interior_d12', center), 74, atol=1e-7)
assert np.isclose(ring_solutions['observed']['cost'], 1.8, atol=1e-6)
assert np.isclose(engine.true_mean('interior_d12', ring_solutions['observed']['x']),88.4,atol=1e-7)

### Figure 2 — Why a perfect historical fit can miss a crucial response pattern
**Manuscript:** §3, Figure 2; §7.1.

Panel A shows the ring of recorded settings and the empty center. Panel B contrasts the fitted response **86** at the chosen center with the stipulated synthetic response **74** (a 12-unit failure below the required level). The observed-setting comparator is modeled as more expensive but has known quality **88.4** in this construction.

In [ ]:
# Step 2.2 — Recreate the scatter/hull view and the fitted-versus-known response curve.
# Figure 2 does not feed the known response into the optimizer. The known curve
# appears solely as a post-decision educational evaluation of the constructed example.

d=pd.read_csv(S/'03_interior_gap_noiseless.csv')
u=d[['x1','x2']].drop_duplicates().to_numpy()
hull=ConvexHull(u); boundary=u[hull.vertices]
profile=pd.read_csv(T/'figure1_response_slice.csv')
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(7.7,3.7),gridspec_kw={'width_ratios':[1,1.10]})
ax1.add_patch(Polygon(boundary,closed=True,facecolor=LIGHT,edgecolor=TEAL,lw=1.8))
ax1.scatter(u[:,0],u[:,1],s=42,facecolors='white',edgecolors=INK,lw=1.2,zorder=3)
ax1.axhline(.5,ls=':',lw=1.3,color=BLUE)
ax1.scatter([.5],[.5],s=135,marker='X',c=RUST,edgecolors='white',lw=.65,zorder=5)
ax1.scatter([.8],[.5],s=113,marker='D',c=TEAL,edgecolors='white',lw=.65,zorder=5)
ax1.annotate('Optimized q',(.5,.5),xytext=(.21,.66),arrowprops={'arrowstyle':'->','color':RUST,'lw':1.3},fontsize=10,color=INK)
ax1.annotate('Observed p',(.8,.5),xytext=(.56,.32),arrowprops={'arrowstyle':'->','color':TEAL,'lw':1.3},fontsize=10,color=INK)
ax1.set(xlim=(.14,.91),ylim=(.14,.89),xlabel=r'Input $x_1$',ylabel=r'Input $x_2$',title='A. Empty interior')
ax1.set_aspect('equal');ax1.grid(color=GRID,lw=.7)
ax2.plot(profile.x1,profile.fitted_response,color=BLUE,ls='--',lw=2.1,label='Fitted response')
ax2.plot(profile.x1,profile.known_synthetic_response,color=INK,lw=2.5,label='Known synthetic response')
ax2.axhline(86,color=INK,lw=1.1,ls=':',alpha=.7)
ax2.scatter([.5,.5,.8],[86,74,88.4],s=[63,110,90],marker='o',color=[BLUE,RUST,TEAL],zorder=5,edgecolors='white',lw=.65)
ax2.annotate('',xy=(.5,74.6),xytext=(.5,85.4),arrowprops={'arrowstyle':'<->','color':RUST,'lw':1.8})
ax2.text(.535,79.3,'12-point\nshortfall',fontsize=10,color=INK,fontweight='bold',va='center')
ax2.annotate('86 predicted',(.5,86),xytext=(.34,90),color=INK,fontsize=10,arrowprops={'arrowstyle':'-','color':BLUE})
ax2.annotate('74 known',(.5,74),xytext=(.21,72.7),color=INK,fontsize=10,arrowprops={'arrowstyle':'-','color':RUST})
ax2.annotate('88.4 known',(.8,88.4),xytext=(.62,91.2),color=INK,fontsize=10,arrowprops={'arrowstyle':'-','color':TEAL})
ax2.set(xlim=(.16,.88),ylim=(71,93),xlabel=r'Input $x_1$ at $x_2=0.5$',ylabel='Quality',title='B. Predicted versus known')
ax2.grid(color=GRID,lw=.7)
fig.subplots_adjust(wspace=.35,bottom=.20,top=.83)
export(fig,'Figure_02_Interior_Gap')

## 3. Five-point targeted follow-up
**Manuscript:** §3.3, §7.1, Appendix A1; Figure 8.

This experiment adds five noiseless observations **at and near the previously unsampled center**, retains all original 48 response records, and refits the **same linear-regression specification once**. It distinguishes discovering the original model's weakness from successfully correcting it. The algorithm may find a different optimum without meeting the known quality requirement; one refit is not a validation procedure.

In [ ]:
# Step 3.1 — Rerun the explicitly declared, fixed five-location sampling experiment.
# The five locations were selected as a prespecified stencil, not tuned after
# examining their responses. This prevents false claims of adaptive convergence.
from targeted_followup import run_targeted_followup
follow = run_targeted_followup(RESULTS / 'targeted_followup', api=engine)
print('Five new settings and their generated, known responses:')
display(follow['new_records'].round(5))
print('Regression diagnostics before and after one refit:')
display(follow['fits'].round(5))
print('Decision consequences (including adjusted-model infeasibility):')
display(follow['results'][['stage','domain','allowance_id','feasible','modeled_cost',
                           'prediction','oracle_mean','allowance']].round(4))

# Check the manuscript's key numerical statements rather than assuming them.
after = follow['results'][(follow['results'].stage=='after_one_refit') &
                          (follow['results'].domain=='hull') &
                          (follow['results'].allowance_id=='zero')].iloc[0]
assert np.isclose(after['oracle_mean'], 80.36, atol=.02)
assert np.isclose(follow['fits'].iloc[-1].mean_pi_half_width, 6.7066, atol=.001)
assert follow['summary']['all_added_points_inside_original_hull']
print('New quality remains below target; original hull geometry did not change.')

## 4. Carpet formulation, proximity, and modeled cost
**Manuscript:** §4 and Figures 3–4; Equation (5) and Table 2.

Two separate questions must not be confused: (1) Does the combination lie within the **joint hull**? (2) How close is it to an actual, **complete** observed formulation? Distances below are Euclidean distances after scaling each chemical coordinate by its own historical range (2.0, 3.5 and 0.13). The resulting distances are descriptive; no process-specific adequate-distance threshold has been established.

In [ ]:
# Step 4.1 — Nearest complete setting to each candidate, with range scaling.
# Scaling is necessary because x3 has a much smaller numerical range than x1 and x2.
# Do not interpret these geometric distances as probabilities of product stability.
def nearest_complete_setting(candidate):
    scaled_offsets = (unique_X - candidate) / original_ranges
    distances = np.sqrt(np.sum(scaled_offsets**2,axis=1))
    idx = int(np.argmin(distances))
    return unique_X[idx], float(distances[idx])

proximity_rows=[]
for label in ('box','hull'):
    p, dist = nearest_complete_setting(carpet_solutions[label]['settings'])
    proximity_rows.append({'formulation':label,'nearest_recorded_setting':tuple(p),
                           'scaled_distance':dist,'modeled_cost':carpet_solutions[label]['cost']})
proximity_table=pd.DataFrame(proximity_rows)
display(proximity_table.round(5))
assert np.isclose(proximity_rows[0]['scaled_distance'],.4112,atol=.0001)
assert np.isclose(proximity_rows[1]['scaled_distance'],.1900,atol=.0001)

### Figure 3 — Why marginally feasible does not mean jointly observed
**Manuscript:** §4, Figure 3.

The three panels show pairwise projections of historical carpet settings. The **box recommendation** is outside the empirical joint hull, while the **hull recommendation** satisfies the shared-weight membership constraint. Remember that being inside every *pairwise* projection does not guarantee three-dimensional hull membership; the full-dimensional solver and weight reconstruction above perform that check.

In [ ]:
# Step 4.2 — Produce three pairwise ingredient-space projections.
# Axes and figure titles use black for journal-print legibility.

df=pd.read_csv(P/'carpet_simulated_data.csv')
U=df[['x1','x2','x3']].drop_duplicates().to_numpy()
main=pd.read_csv(T/'carpet_main_table.csv')
box=main.loc[main.formulation=='Individual bounds',['x1','x2','x3']].iloc[0].to_numpy(dtype=float)
hullc=main.loc[main.formulation=='Joint hull',['x1','x2','x3']].iloc[0].to_numpy(dtype=float)
fig,axs=plt.subplots(1,3,figsize=(7.75,3.4))
for j,(ia,ib) in enumerate([(0,1),(0,2),(1,2)]):
    a=axs[j]
    xy=U[:,[ia,ib]];poly=xy[ConvexHull(xy).vertices];poly=np.vstack([poly,poly[0]])
    a.fill(poly[:,0],poly[:,1],alpha=.10,color=TEAL)
    a.plot(poly[:,0],poly[:,1],color=TEAL,lw=1.9)
    a.scatter(xy[:,0],xy[:,1],color=INK,alpha=.65,s=21)
    for point,marker,col in [(box,'X',RUST),(hullc,'D',BLUE)]:
        a.scatter([point[ia]],[point[ib]],marker=marker,s=115,color=col,
                  edgecolors='white',lw=.7,zorder=4)
    a.set(xlabel=f'$x_{ia+1}$',ylabel=f'$x_{ib+1}$',title=f'{"ABC"[j]}. $x_{ia+1}$ vs $x_{ib+1}$')
    a.grid(color=GRID,lw=.7)
    a.tick_params(axis='both',labelsize=10)
fig.legend(handles=[Line2D([0],[0],marker='o',color='none',markerfacecolor=INK,markersize=6,label='Observed'),
                    Line2D([0],[0],marker='X',color='none',markerfacecolor=RUST,markersize=8,label='Box solution'),
                    Line2D([0],[0],marker='D',color='none',markerfacecolor=BLUE,markersize=8,label='Hull solution')],
           loc='lower center',ncol=3,frameon=False,bbox_to_anchor=(.5,.01),fontsize=10)
fig.subplots_adjust(left=.075,right=.98,bottom=.27,top=.83,wspace=.35)
export(fig,'Figure_03_Carpet_Joint_Support')

## 5. Response uncertainty and allowance sensitivity
**Manuscript:** §5.1, Appendix A2, Figure 5 and Table 2.

We compute the classical **two-sided 95% prediction-interval half-width** at each of the 150 original input rows and average them to form a **constant illustrative margin** \(m\). We then solve the hull-constrained model at \(m\in\{0,0.5,1,1.4358,2,3\}\). These are *modeled sensitivity comparisons*, not a calibrated probability guarantee for the point selected **after optimization**. The first three unbuffered solutions in Table 2 have fitted stability at the 80% threshold; they do not guarantee a future observed stability of 80%.

In [ ]:
# Step 5.1 — Build classical pointwise prediction half-widths from the OLS design.
# For a new response at x, the leading 1 in sqrt(1 + leverage) accounts for
# new-observation noise; leverage captures uncertainty in the estimated mean.
# The *average* historical half-width is deliberately kept constant for every
# candidate during optimization, which differs from optimizing a pointwise bound.
df_error = len(X) - len(carpet_beta)
residual_sd = np.sqrt(np.dot(residuals,residuals)/df_error)
XTX_inv = np.linalg.inv(design_matrix.T @ design_matrix)
quantile = t.ppf(.975, df_error)
leverage = np.einsum('ij,jk,ik->i',design_matrix,XTX_inv,design_matrix)
half_widths = quantile*residual_sd*np.sqrt(1+leverage)
mean_historical_margin = float(half_widths.mean())
print('Average historical two-sided 95% PI half-width:', mean_historical_margin)
assert np.isclose(mean_historical_margin,1.43576267,atol=1e-6)

# Re-solve the same hull problem at each explicitly stated allowance.
allowances = [0, .5, 1.0, mean_historical_margin, 2.0, 3.0]
allowance_solutions={}
allowance_rows=[]
for margin in allowances:
    candidate=solve_lp(carpet_beta,carpet_costs,X,'min_cost','hull',
                       target=quality_target+margin, budget=budget,
                       fixed_box=(original_lower,original_upper))
    assert candidate['feasible']
    allowance_solutions[margin]=candidate
    neighbor, distance=nearest_complete_setting(candidate['settings'])
    allowance_rows.append({'allowance':margin,'required_fitted_stability':quality_target+margin,
                           'modeled_cost':candidate['cost'], 'fitted_stability':candidate['prediction'],
                           'nearest_distance':distance, **{f'x{i+1}':v for i,v in enumerate(candidate['settings'])}})
allowance_table=pd.DataFrame(allowance_rows)
display(allowance_table.round(5))

buffered_solution=allowance_solutions[mean_historical_margin]
_, buffered_distance=nearest_complete_setting(buffered_solution['settings'])
assert np.isclose(buffered_solution['cost'],34.16,atol=.01)
assert np.isclose(buffered_distance,.0970,atol=.0002)
print('Modeled box-to-hull premium:',
      f"{100*(carpet_solutions['hull']['cost']/carpet_solutions['box']['cost']-1):.2f}%")

### Figure 4 — Cost versus proximity for three carpet candidates
**Manuscript:** §4, Figure 4.

This graphic places model cost and nearest-setting distance side-by-side. It shows a transparent modeled tradeoff, **not** an estimated manufacturing risk reduction.

In [ ]:
# Step 5.2 — Export the comparison of cost and distance using verified exhibit inputs.
# The underlying distances can be recomputed with nearest_complete_setting above.

prox=pd.read_csv(T/'carpet_proximity_table.csv')
labels=['Bounds only','Joint hull','Hull + buffer']
costs=[main.loc[main.formulation==n,'modeled_cost'].iloc[0]
       for n in ['Individual bounds','Joint hull','Joint hull plus historical allowance']]
dists=[prox.loc[prox.candidate==n,'range_standardized_distance'].iloc[0]
       for n in ['box','hull','hull_historical_allowance']]
cols=[RUST,BLUE,TEAL];y=np.arange(3)
fig,(a,b)=plt.subplots(1,2,figsize=(7.7,3.05),sharey=True)
for axis, vals, xmin,xmax,fmt,title in [
    (a,costs,31.7,34.9,'${:.2f}','A. Modeled cost ($)'),
    (b,dists,0,.49,'{:.3f}','B. Nearest-setting distance')]:
    axis.set_xlim(xmin,xmax);axis.set_yticks(y,labels);axis.set_ylim(2.5,-.5)
    axis.grid(axis='x',color=GRID,lw=.8);axis.tick_params(axis='y',length=0)
    for i,v in enumerate(vals):
        axis.plot([xmin,v],[i,i],color=cols[i],alpha=.42,lw=3.2)
        axis.scatter(v,i,s=100,c=cols[i],edgecolors='white',zorder=3,lw=.8)
        offset=(xmax-xmin)*.035
        tx=min(v+offset,xmax-(xmax-xmin)*.12)
        if tx < v: tx=v-(xmax-xmin)*.04;ha='right'
        else: ha='left'
        axis.text(tx,i-.18,fmt.format(v),fontsize=10.7,color=INK,
                  fontweight='bold',ha=ha)
    axis.set_title(title,pad=13)
    axis.spines['left'].set_visible(False)
fig.text(.51,.048,'Lower distance indicates closer recorded settings—not physical admissibility.',
         ha='center',fontsize=9.8,color=INK)
fig.subplots_adjust(left=.23,right=.97,bottom=.25,top=.79,wspace=.31)
export(fig,'Figure_04_Carpet_Cost_Proximity')

### Figure 5 — What a constant quality margin costs in the model
**Manuscript:** §5.1, Figure 5.

Figure 5 uses full-precision modeled-cost results. A steeper curve would show that additional margin becomes expensive in the optimization model; **it would still not establish actual coverage**. The calculation above verifies the historical margin and each point on the curve.

In [ ]:
# Step 5.3 — Recreate the six-scenario allowance-versus-modeled-cost curve.
# The reference CSV retains the original full-precision solver outputs.

sweep=pd.read_csv(O/'carpet_allowance_sweep.csv')
fig,ax=plt.subplots(figsize=(7.2,3.35))
ax.plot(sweep.allowance,sweep.modeled_cost,color=BLUE,lw=2.5,marker='o',markersize=6.8)
focus=sweep.loc[np.isclose(sweep.allowance,1.43576267,atol=1e-4)].iloc[0]
ax.scatter([focus.allowance],[focus.modeled_cost],s=146,color=TEAL,edgecolors='white',lw=.8,zorder=6)
ax.annotate('Historical mean half-width\n$34.16 at 1.436 points',
            (focus.allowance,focus.modeled_cost),xytext=(1.85,33.52),fontsize=10.2,
            color=INK,arrowprops={'arrowstyle':'->','color':TEAL,'lw':1.35})
ax.annotate('No allowance: $33.27',(0,33.266886),xytext=(.08,33.02),fontsize=10.2,color=INK)
ax.annotate('3 points: $35.13',(3,35.125),xytext=(2.0,35.23),fontsize=10.2,color=INK)
ax.set(xlim=(-.16,3.30),ylim=(32.91,35.57),
       xlabel='Constant allowance (quality percentage points)',ylabel='Modeled ingredient cost ($)')
ax.grid(color=GRID,lw=.8)
fig.subplots_adjust(left=.12,right=.97,bottom=.21,top=.94)
export(fig,'Figure_05_Allowance_Cost')

## 6. Matched controls and repeated noisy simulations
**Manuscript:** §5.2, Appendix A3, Table 3 and Figure 6.

The paired **linear-response control** and **correlated-input** scenario have exactly the same training observations, hence the same fitted regression and the same optimization decisions. Their true responses differ only at certain *unobserved* combinations. This identifies when a hull restriction can prevent an off-support response loss and when it merely increases cost without improving the true mean.

The larger screen uses **eight configurations × (two representative cases + 100 noisy replications)**, producing 816 configuration-run instances and 16,320 optimization comparisons. The paper's Monte Carlo table is read from the **archived full study results** by default, not silently rerun. The optional full screen at the end can regenerate it.

In [ ]:
# Step 6.1 — Verify the crucial matched-control design at the noiseless display seed.
control = engine.generate_dataset('control_w005', noise_sd=0.0, seed=engine.REPRESENTATIVE_SEED)
correlated = engine.generate_dataset('correlated_w005', noise_sd=0.0, seed=engine.REPRESENTATIVE_SEED)
assert np.array_equal(control['X'],correlated['X'])
assert np.allclose(control['y'],correlated['y'],atol=1e-12)
control_fit=engine.fit_ols(control['X'],control['y'])
correlated_fit=engine.fit_ols(correlated['X'],correlated['y'])
assert np.allclose(control_fit['beta'],correlated_fit['beta'],atol=1e-12)

# The same trained model selects the same points, but known quality can differ.
matched_rows=[]
for domain in ('box','hull'):
    sol=engine.solve_model(control_fit,control['X'],'control_w005',domain=domain)
    truth_control=float(engine.true_mean('control_w005',sol['x']))
    truth_penalty=float(engine.true_mean('correlated_w005',sol['x']))
    matched_rows.append({'domain':domain,'x1':sol['x'][0],'x2':sol['x'][1],
                         'modeled_cost':sol['cost'],'known_control_quality':truth_control,
                         'known_correlated_quality':truth_penalty})
matched_table=pd.DataFrame(matched_rows)
display(matched_table.round(5))
assert np.isclose(matched_table.loc[matched_table.domain=='box','modeled_cost'].iloc[0],1.05,atol=1e-6)
assert np.isclose(matched_table.loc[matched_table.domain=='hull','modeled_cost'].iloc[0],1.45,atol=1e-6)

In [ ]:
# Step 6.2 — Load, label, and validate the precomputed 100-repetition screening results.
SCREEN = WORK / 'synthetic_study' / 'screen'
with (SCREEN / 'screen_facts.json').open() as f:
    screen_facts = json.load(f)
full_screen_results = pd.read_csv(SCREEN / 'optimization_results.csv')
full_screen_fits = pd.read_csv(SCREEN / 'model_fits.csv')
full_screen_summary = pd.read_csv(SCREEN / 'replication_summary.csv')
noise_table = pd.read_csv(WORK/'manuscript_exhibits/tables/noise_main_table.csv')

# Every number below describes archive contents; no new repetitions are run here.
assert len(full_screen_results) == 16320
assert len(full_screen_fits) == 816
assert full_screen_fits.training_sha256.nunique() == 510
assert screen_facts['n_support_omissions'] == 5304
assert len(noise_table) == 9
print('Archived runs:',len(full_screen_fits),'| unique training tables:',
      full_screen_fits.training_sha256.nunique())
print('Archived optimization comparisons:',len(full_screen_results))
print('Archived synthetic setting omissions:',screen_facts['n_support_omissions'])
print('Paper Table 3 (saved 100-fit comparison):')
display(noise_table)

### Figure 6 — Two different failure measures
**Manuscript:** §5.2, Figure 6 and Table 3.

Panel A counts how often the **known conditional mean** at the selected decision is below the target across 100 independently generated noisy fits. Panel B averages the **conditional probability that one future noisy response** falls below the target. They are not interchangeable: a decision can have a mean at the threshold yet still face substantial future-outcome variability. Error bars in Panel B describe simulation uncertainty across fits, not predictive coverage for an optimized manufacturing decision.

In [ ]:
# Step 6.3 — Plot only the nine prespecified scenarios in Table 3.
# The plotted means and Monte Carlo errors come from archived screen results.
# The following cell does NOT regenerate the 816 scenario-run screen.

n=pd.read_csv(T/'noise_main_table.csv')
n['mean_shortfall_pct']=n.means_below_target.str.split('/').str[0].astype(float)
labels=[]
for _,r in n.iterrows():
    scenario={'Affine control':'Affine','Correlated inputs':'Correlated','Interior gap':'Interior gap'}[r.scenario]
    dom={'Box':'Box','Hull':'Hull','Observed settings':'Observed'}[r.domain]
    allow=' + buffer' if pd.notna(r.allowance) else ''
    labels.append(f'{scenario}: {dom}{allow}')
ys=np.arange(len(n)); colors=[INK if c=='Affine control' else TEAL if c=='Correlated inputs' else RUST for c in n.scenario]
fig,(ax_a,ax_b)=plt.subplots(2,1,figsize=(7.65,6.45))
for ax,values,title,format_digits in [
    (ax_a,n.mean_shortfall_pct.to_numpy(float),'A. Selected known mean below the target',0),
    (ax_b,n.mean_future_failure_pct.to_numpy(float),'B. Average probability of a future response below target',1)]:
    ax.set_xlim(-2.5,110);ax.set_ylim(len(n)-.5,-.5)
    ax.set_yticks(ys,labels);ax.tick_params(axis='y',length=0,labelsize=9.6)
    ax.axvline(50,ls=':',color='#777777',lw=1)
    for sep in [2.5,5.5]:ax.axhline(sep,lw=1.0,color='#ABB8C0')
    for i,(v,c) in enumerate(zip(values,colors)):
        ax.scatter(v,i,color=c,s=72,zorder=3,edgecolors='white',lw=.65)
        if ax is ax_b:
            err=float(n.mcse_percentage_points.iloc[i])
            ax.plot([max(0,v-err),min(100,v+err)],[i,i],color=c,lw=2,zorder=2)
        tx=v-4.0 if v>=91 else v+3.8
        ha='right' if v>=91 else 'left'
        text=f'{v:.0f}%' if format_digits==0 else f'{v:.1f}%'
        ax.text(tx,i-.13,text,fontsize=9.9,color=INK,ha=ha,va='center',fontweight='bold')
    ax.set_title(title,pad=9,loc='left')
    ax.set_xticks([0,25,50,75,100]);ax.grid(axis='x',color=GRID,lw=.75)
    ax.set_xlabel('% across 100 fitted datasets' if ax is ax_a else 'Mean percentage; whiskers = ±1 Monte Carlo SE')
fig.subplots_adjust(left=.32,right=.96,bottom=.09,top=.95,hspace=.35)
export(fig,'Figure_06_Simulation_Contrasts')

## 7. Boundary-setting sensitivity
**Manuscript:** §6, Appendix A4, Figure 7.

This exercise removes **one distinct complete input setting from the hull at a time** while holding the carpet regression fit to all 150 original response records. That is crucial: this is a test of **search-domain dependence**, not regression cross-validation and not a justification for discarding legitimate records. We quantify modeled-cost change and movement of the ingredient settings relative to their observed ranges.

In [ ]:
# Step 7.1 — Recompute all 49 carpet-setting omissions, keeping beta fixed.
# Repeated response records at one input combination are omitted from the
# geometry as ONE setting, but are retained in the previously fitted regression.
original_beta = carpet_beta.copy()
base_hull = carpet_solutions['hull']
base_cost, base_x = base_hull['cost'],base_hull['settings']
omission_rows=[]
for setting in unique_X:
    new_points = unique_X[~np.all(np.isclose(unique_X,setting,atol=1e-12,rtol=0),axis=1)]
    candidate=solve_lp(carpet_beta,carpet_costs,X,'min_cost','hull',
                       target=quality_target,budget=budget,
                       domain_points=new_points,
                       fixed_box=(original_lower,original_upper))
    assert candidate['feasible']
    delta=candidate['settings']-base_x
    omission_rows.append({'removed_setting':tuple(setting),'feasible':candidate['feasible'],
                          'modeled_cost':candidate['cost'],'cost_increase':candidate['cost']-base_cost,
                          'cost_change_pct':100*(candidate['cost']-base_cost)/base_cost,
                          'largest_scaled_coordinate_movement':float(np.max(np.abs(delta/original_ranges)))})
omission_table = pd.DataFrame(omission_rows)
assert np.array_equal(carpet_beta,original_beta), 'Regression must be held fixed.'
assert len(omission_table)==49 and omission_table.feasible.all()
assert int((omission_table.cost_increase>1e-7).sum()) == 2
assert np.isclose(omission_table.largest_scaled_coordinate_movement.max(),.009914275,atol=1e-6)
print('Changes above 1e-7:',int((omission_table.cost_increase>1e-7).sum()))
display(omission_table[omission_table.cost_increase>1e-7].round(7))

### Figure 7 — Quantify the sensitivity, do not just report the feasible hull
**Manuscript:** §6, Figure 7.

This figure shows the 47 nearly unchanged cases and the two small modeled-cost increases; every one of the 49 reduced-hull models remained feasible. It should not be read as validation of the chemical response or of the manufacturing constraints.

In [ ]:
# Step 7.2 — Figure 7 uses the full archived sensitivity CSV as an audit target.
# The recomputed omission_table above independently checks its two nonzero changes.

ss=pd.read_csv(O/'carpet_support_sensitivity.csv')
positive=ss.loc[ss.cost_change>1e-7].sort_values('cost_change',ascending=False)
fig,ax=plt.subplots(figsize=(7.5,3.1))
ax.axvline(0,color=INK,lw=1)
row_names=[f"Remove ({r.removed_x1:.1f}, {r.removed_x2:.1f}, {r.removed_x3:.2f})"
           for _,r in positive.iterrows()]+['Other 47 settings']
vals=positive.cost_change.tolist()+[0]
for y,(v,name) in enumerate(zip(vals,row_names)):
    col=RUST if y==0 else BLUE if y==1 else INK
    ax.scatter(v,y,color=col,s=104 if y<2 else 70,zorder=4,edgecolors='white',lw=.8)
    ax.plot([0,v],[y,y],color=col,lw=2.7,alpha=.6)
    ax.text(v+.00045,y-.15,f'${v:.4f}' if y<2 else '$0.0000',
            fontsize=10.6,color=INK,fontweight='bold')
ax.set(yticks=np.arange(3),yticklabels=row_names,xlim=(-.0005,.0170),ylim=(2.5,-.5),
       xlabel='Modeled cost increase after omitting a complete setting ($)')
ax.set_xticks([0,.005,.010,.015]);ax.grid(axis='x',color=GRID,lw=.8);ax.tick_params(axis='y',length=0)
ax.text(.98,.03,'49 feasible • 47 unchanged • 2 changes below 0.04%',
        ha='right',va='bottom',transform=ax.transAxes,fontsize=9.7,color=INK)
fig.subplots_adjust(left=.35,right=.96,bottom=.24,top=.95)
export(fig,'Figure_07_Boundary_Perturbation')

## 8. Decision records and editable PowerPoint diagrams
**Manuscript:** §7.1 (Figure 8) and §7.2 (Figure 9).

The paper's final step is to turn statistical and optimization diagnostics into **conditional recommendations**. Figure 8 asks which synthetic candidate should be tested and why new observations matter; Figure 9 contrasts three modeled carpet recipes and the evidence required before a manufacturing trial.

The following cell runs the original Python drawing scripts to create **native editable PowerPoint shapes**. The embedded archive also includes the verified high-resolution PNG previews that came from rendering these presentations. Colab can generate the `.pptx` files without LibreOffice. If a LibreOffice executable is present, you may optionally re-render PNG previews from the new slides; the code does not need a proprietary image service.

In [ ]:
# Step 8.1 — Rebuild Figure 8 and Figure 9 as fully editable .pptx files.
# Each PPTX shape, line, circle, label, arrow, and card is created by python-pptx.
# No generative-image model or screenshot manipulation is involved.
for section, filename in [('§7.1', 'build_figure8_editable_ppt.py'),
                          ('§7.2', 'build_figure9_editable_ppt.py')]:
    script = WORK / 'code' / filename
    runpy.run_path(str(script),run_name='__main__')
    print('Recreated editable drawing for', section)

# The scripts write beside their own source files. Copy final PPTX assets to
# the same figures directory as the PNGs/SVGs produced by Matplotlib.
for number, stem in [(8,'Two_Input_Decision_Record'),(9,'Carpet_Decision_Record')]:
    file = WORK / 'code' / f'Figure_{number:02d}_{stem}.pptx'
    assert file.exists()
    shutil.copy2(file, FIGURES / file.name)
    print('PowerPoint created:',FIGURES/file.name)

# Preview images were rendered from these PowerPoint sources in the original
# verified companion. They are included here because python-pptx does not itself
# contain a slide-to-PNG renderer; the editable PPTX is recreated above.
for number, stem in [(8,'Two_Input_Decision_Record'),(9,'Carpet_Decision_Record')]:
    preview = FIGURES / f'Figure_{number:02d}_{stem}.png'
    assert preview.exists(), f'Missing preview: {preview}'
    display(Image(filename=str(preview),width=720))

## 9. Final integrity checks and file exports
**Manuscript:** entire paper, particularly Tables 1–3 and Figures 1–9.

This cell verifies all nine manuscript figure files exist, checks that edited PowerPoint sources were regenerated, and exports a CSV for each central model comparison and a ZIP with **current notebook-generated analytical figures and editable PowerPoint files**. It does not claim field validation or new manufacturing outcomes.

In [ ]:
# Step 9.1 — Check important model values explicitly before exporting.
# Do not relax numerical tolerances simply to force a false match.
assert np.isclose(ring_fit['r_squared'],1.0,atol=1e-10)
assert np.isclose(carpet_r2,.988708,atol=5e-6)
assert np.isclose(mean_historical_margin,1.43576267,atol=1e-6)
assert np.isclose(buffered_solution['cost'],34.16,atol=.01)
assert (omission_table.cost_increase>1e-7).sum()==2
assert len(full_screen_fits)==816 and len(full_screen_results)==16320

# Figures 1–7 each have newly exported PNG and vector SVG versions.
for number,stem in [(1,'Evidence_Workflow'),(2,'Interior_Gap'),(3,'Carpet_Joint_Support'),
                    (4,'Carpet_Cost_Proximity'),(5,'Allowance_Cost'),
                    (6,'Simulation_Contrasts'),(7,'Boundary_Perturbation')]:
    for ext in ('png','svg'):
        assert (FIGURES / f'Figure_{number:02d}_{stem}.{ext}').exists()

# Figures 8–9 have regenerated editable .pptx and verified original PNG previews.
for number,stem in [(8,'Two_Input_Decision_Record'),(9,'Carpet_Decision_Record')]:
    for ext in ('png','pptx'):
        assert (FIGURES / f'Figure_{number:02d}_{stem}.{ext}').exists()

# Step 9.2 — Export user-friendly CSVs for the manuscript's primary comparisons.
EXPORTS = WORK / 'manuscript_outputs'
EXPORTS.mkdir(exist_ok=True)
for name,frame in [('Table_1_Synthetic_Domains',ring_comparison),
                   ('Table_2_Carpet_Candidates',carpet_baseline_table),
                   ('Carpet_Allowance_Sensitivity',allowance_table),
                   ('Carpet_Nearest_Setting_Distances',proximity_table),
                   ('Matched_Synthetic_Controls',matched_table),
                   ('Table_3_Archived_100_Replication_Comparison',noise_table),
                   ('Carpet_49_Hull_Setting_Omissions',omission_table)]:
    frame.to_csv(EXPORTS/f'{name}.csv',index=False,float_format='%.15g')

# Step 9.3 — Make a single portable ZIP with figures, editable PPTX and CSVs.
final_zip = WORK / 'IJAA_Colab_Recreated_Exhibits.zip'
with zipfile.ZipFile(final_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for folder in (FIGURES, EXPORTS):
        for path in sorted(folder.rglob('*')):
            if path.is_file():
                z.write(path,arcname=path.relative_to(WORK))
print('All checks passed.')
print('Figures (PNG):',len(list(FIGURES.glob('Figure_*.png'))))
print('Editable PowerPoint figures:',len(list(FIGURES.glob('Figure_*.pptx'))))
print('Export ZIP:',final_zip,'| bytes:',final_zip.stat().st_size)

### Optional: download outputs directly from Colab

After **Runtime → Run all**, the ZIP is available in the left-hand Colab Files panel under `IJAA_Tutorial_Colab`. The cell below uses Colab's download dialog when available; it simply prints the path in a local Jupyter session.

In [ ]:
# Step 9.4 — Trigger a download only on Google Colab (safe in local Jupyter).
if IN_COLAB:
    from google.colab import files
    print('Downloading:',final_zip.name)
    files.download(str(final_zip))
else:
    print('Download or copy:',final_zip)

## 10. Optional: regenerate the entire noisy simulation screen from scratch
**Manuscript:** §5.2, Appendix A3 (100 noisy repetitions for each of eight designs).

**Default: OFF.** The archived full-screen results used above are reproducible and fully included inside this notebook. To rerun all original random seeds, set `RUN_FULL_SCREEN = True` and execute this cell. It will create a new `full_screen_rerun/` directory, regenerate all fit and optimization CSVs, and check the declared study totals. This is a substantive recomputation, unlike the default archive-reading cells above.

To try a shorter *test* before the complete design, set `RUN_TEST_SCREEN = True` and leave `RUN_FULL_SCREEN = False`; it will rerun just a few declared configurations and seeds and label the results as **partial**, not as Table 3 reproduction. Never mix the partial counts with the full manuscript counts.

In [ ]:
# Step 10.1 — Optional complete replication of the prespecified synthetic experiment.
RUN_FULL_SCREEN = False  # Change to True only when you want all 816 configuration runs.
RUN_TEST_SCREEN = False  # Small coding check; never treated as final paper results.

if RUN_FULL_SCREEN:
    # This calls the same generative mechanisms, random-seed sequence, OLS fits,
    # domain-constrained LPs, and 100-repetition summaries as the paper.
    fresh_screen = engine.run_screen(RESULTS/'full_screen_rerun',verbose=True)
    facts = fresh_screen['facts']
    assert facts['n_configuration_run_instances'] == 816
    assert facts['n_unique_training_datasets'] == 510
    assert facts['n_optimization_comparisons'] == 16320
    assert facts['n_support_omissions'] == 5304
    print('Full synthetic screen regenerated and counts verified:',facts)
elif RUN_TEST_SCREEN:
    partial = engine.run_screen(RESULTS/'partial_test_only',
                 configs=['control_w005','interior_d12'],
                 replication_seeds=(100,101),verbose=False)
    print('Partial test only (NOT the full study):',partial['facts'])
else:
    print('Full re-simulation intentionally skipped; default figures use verified archived screen.')

## Interpretation, source provenance, and manuscript alignment

- **Sections 2–4 / Appendix A1:** The synthetic example is a demonstration of an *optimizer-selected failure inside a convex hull*, not a claim that convex hulls are always inadequate. The carpet benchmark measures modeled responses/costs; proprietary chemical identities, exact units, physical operating limits, and manufacturer validation trials are not available.
- **Section 5.1 / Appendix A2:** The constant margin derived from average historical prediction-interval half-widths is a **heuristic sensitivity scenario**, not calibrated coverage after optimizing over inputs.
- **Section 5.2 / Appendix A3:** The matched controls show that a costlier hull restriction can improve quality in one stipulated mechanism but only add cost in another. Future-outcome probability is distinct from a conditional-mean failure count.
- **Section 6 / Appendix A4:** One-setting omission changes the geometric feasible region while holding the regression fit fixed. A small change in modeled cost is not a guarantee of stable production quality.
- **Sections 7.1–7.2:** Figures 8–9 turn diagnostics into decisions requiring independent evidence; neither is an operational approval.

**Literature referenced in the manuscript:** Bertsimas and Kallus (2020), *Management Science*, doi:10.1287/mnsc.2018.3253; Boyd and Vandenberghe (2004), *Convex Optimization*, doi:10.1017/CBO9780511804441; Smith and Winkler (2006), *Management Science*, doi:10.1287/mnsc.1050.0451; Heckert et al. (2002), *NIST/SEMATECH e-Handbook of Statistical Methods*, doi:10.18434/M32189; Morris, White, and Crowther (2019), *Statistics in Medicine*, doi:10.1002/sim.8086. These are methodological context and are not represented as new validation of the manufacturing benchmark.